In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import features
import last_items

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100
    new_split = "_new_split"


config = Config()

In [4]:
mod = "popular_at_day"

In [5]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = df_test[df_test[config.col_aid] != pad]
# df_test = pl.from_pandas(df_test)
# df_test = df_test.groupby([config.col_user]).agg(
#     [
#         pl.col(config.col_aid).list().alias(config.col_train_seq),
#     ]
# )

# data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()

# recommender = last_items.LastItemsRecommender()
# recs = recommender.recommend(data_to_predict, n_recs=config.num_recs)

# df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
# df_idx["user"] = df_test.to_pandas()["user"].astype(int)
# df_idx = pl.from_pandas(df_idx)
# recs = recs.join(df_idx, on="index")

# recs = recs.rename({'similar_rank': f'last_items_rank'})

# recs.write_parquet(config.cur_model_output_path + f'cur_{mod}_df_test_preds_{config.num_recs}.parquet')

CPU times: user 6 s, sys: 1.2 s, total: 7.2 s
Wall time: 459 ms


In [6]:
%%time

df_test = df_test.to_pandas()
df_test = utils.make_date_stats(df_test)
df_test = df_test.drop('date_day', axis=1)
df_test = pl.from_pandas(df_test)

CPU times: user 3.23 s, sys: 517 ms, total: 3.75 s
Wall time: 3.65 s


In [7]:
df_test.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour
i64,i64,i64,str,i64,datetime[ns],i8,i8,i8,i8
12899779,59625,1661724000278,"""clicks""",0,2022-08-29 00:00:00.278000128,29,35,0,0
12899780,1142000,1661724000378,"""clicks""",0,2022-08-29 00:00:00.377999872,29,35,0,0
12899781,141736,1661724000559,"""clicks""",0,2022-08-29 00:00:00.559000064,29,35,0,0
12899782,1669402,1661724000568,"""clicks""",0,2022-08-29 00:00:00.568,29,35,0,0
12899783,255297,1661724000572,"""clicks""",0,2022-08-29 00:00:00.572,29,35,0,0


In [8]:
%%time

df_stats = features.get_item_stats_per_day(df_test, max_rank=50)
df_test = df_test.sort("ts").unique(subset=["user"], keep="last")[['user', 'n_day']]
recs = df_test.join(df_stats, on='n_day').rename({'rank': 'popular_at_day_rank', 'count': 'popular_at_day_score', 'user': 'session', 'aid': 'item_id'})
recs = recs.drop('n_day')

CPU times: user 14.9 s, sys: 8.9 s, total: 23.8 s
Wall time: 1.73 s


In [9]:
recs

session,item_id,popular_at_day_score,popular_at_day_rank
i64,i64,u32,u32
12899779,1460571,1213,1
12899779,1260564,912,2
12899779,108125,871,3
12899779,1741695,810,4
12899779,1214819,803,5
12899779,1531805,692,6
12899779,187686,690,7
12899779,641250,666,8
12899779,836852,590,9


In [10]:
recs.write_parquet(
    config.cur_model_output_path + f"cur_{mod}_df_test_preds_{config.num_recs}.parquet"
)

In [11]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False, new_split=True)
X_train_, X_val_, X_val_seq_, ordinal_encoder = utils.dfs_interactions_to_list(
    config, 
    X_train, 
    X_val, 
    X_val_seq, 
    ordinal_encoder, 
    return_only_val=True , 
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 11min 41s, sys: 15min 16s, total: 26min 57s
Wall time: 1min 56s


In [12]:
%%time

X_val_seq = X_val_seq.to_pandas()
X_val_seq = utils.make_date_stats(X_val_seq)
X_val_seq = X_val_seq.drop('date_day', axis=1)
X_val_seq = pl.from_pandas(X_val_seq)

CPU times: user 2.38 s, sys: 457 ms, total: 2.84 s
Wall time: 2.61 s


In [13]:
%%time

df_stats = features.get_item_stats_per_day(X_val_seq, max_rank=50)
X_val_seq = X_val_seq.sort("ts").unique(subset=["user"], keep="last")[['user', 'n_day']]
recs = X_val_seq.join(df_stats, on='n_day').rename({'rank': 'popular_at_day_rank', 'count': 'popular_at_day_score', 'user': 'session', 'aid': 'item_id'})
recs = recs.drop('n_day')

CPU times: user 14.6 s, sys: 8.6 s, total: 23.2 s
Wall time: 1.58 s


In [14]:
recs

session,item_id,popular_at_day_score,popular_at_day_rank
i64,i64,u32,u32
12899779,1460571,941,1
12899779,1260564,784,2
12899779,108125,735,3
12899779,1214819,662,4
12899779,187686,652,5
12899779,1741695,573,6
12899779,641250,567,7
12899779,1531805,534,8
12899779,63971,524,9


In [15]:
recs.write_parquet(
    config.cur_model_output_path
    + f"cur_{mod}_df_val_preds_{config.num_recs}{config.new_split}.parquet"
)

In [16]:
X_val_ = X_val_.rename({"user": "session"})

In [17]:
%%time

recs = (
        recs.groupby(["session"])
        .agg(
            [
                pl.col("item_id").list(),
            ]
        )
        .sort("session")
        .to_pandas()
    )

recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))
recs = recs.merge(X_val_[["session", "target"]].to_pandas())

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

print(utils.recall_several_targets(preds, targets))

  0%|          | 0/1671803 [00:00<?, ?it/s]

0.010636784139307107
CPU times: user 1min 21s, sys: 3.74 s, total: 1min 25s
Wall time: 24 s
